In [5]:
using Pkg # package manager
Pkg.activate(".") # activate the current environment (Project.toml)
Pkg.status() # show the status of the current environment

  Activating project at `c:\Users\abdos\OneDrive\Documents\Abdullah\Uni\Algorithms for Data Analsysis Lab\Lab-CA-SS26-resources`


Status `C:\Users\abdos\OneDrive\Documents\Abdullah\Uni\Algorithms for Data Analsysis Lab\Lab-CA-SS26-resources\Project.toml`
  [13f3f980] CairoMakie v0.15.11
⌅ [864edb3b] DataStructures v0.18.22
⌅ [587475ba] Flux v0.14.21
  [1ecd5474] GraphMakie v0.6.3
⌃ [cffab07f] GraphNeuralNetworks v0.6.19
⌃ [86223c79] Graphs v1.13.1
  [e79ef3ae] GraphsOptim v0.1.0
  [87dc4568] HiGHS v1.23.0
  [4076af6c] JuMP v1.30.1
  [eb30cadb] MLDatasets v0.7.21
⌃ [f1d291b0] MLUtils v0.4.8
  [91a5bcdd] Plots v1.41.6
  [10745b16] Statistics v1.11.1
Info Packages marked with ⌃ and ⌅ have new versions available. Those with ⌃ may be upgradable, but those with ⌅ are restricted by compatibility constraints from upgrading. To see why use `status --outdated`


In [6]:
Pkg.add("DataStructures")
Pkg.add("Graphs") # install the Graphs.jl package

   Resolving package versions...
     Project No packages added to or removed from `C:\Users\abdos\OneDrive\Documents\Abdullah\Uni\Algorithms for Data Analsysis Lab\Lab-CA-SS26-resources\Project.toml`
    Manifest No packages added to or removed from `C:\Users\abdos\OneDrive\Documents\Abdullah\Uni\Algorithms for Data Analsysis Lab\Lab-CA-SS26-resources\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\abdos\OneDrive\Documents\Abdullah\Uni\Algorithms for Data Analsysis Lab\Lab-CA-SS26-resources\Project.toml`
    Manifest No packages added to or removed from `C:\Users\abdos\OneDrive\Documents\Abdullah\Uni\Algorithms for Data Analsysis Lab\Lab-CA-SS26-resources\Manifest.toml`


In [7]:
using Graphs
using DataStructures


In [8]:
function canonical_color_refinement(g::AbstractGraph, alpha::Vector{Int}, S::Vector{Int})
    n = nv(g)
    
    length(alpha) == n || throw(ArgumentError("Initial coloring alpha must have length nv(g)"))
    
    # 1. Initialize core data structures (Lines 33-36)
    # Using Sets for O(1) deletion during the split phase.
    C = [Set{Int}() for _ in 1:n] 
    A = [Vector{Int}() for _ in 1:n]
    maxcdeg = zeros(Int, n)
    mincdeg = zeros(Int, n)
    
    # 2. Initialize vertex-specific arrays (Lines 37-40)
    cdeg = zeros(Int, n)
    colour = copy(alpha)
    
    for v in 1:n
        push!(C[alpha[v]], v)
    end
    
    k = maximum(alpha) 
    
    # Asumption sort function is O(n log n) 
    S_sorted = sort(S)
    
    # Initialize stack and the O(1) parallel boolean array
    Srefine = Stack{Int}()
    in_stack = falses(n)
    
    for c in S_sorted
        push!(Srefine, c)
        in_stack[c] = true
    end
    
    # Buffers to prevent allocations inside the loop
    Colorsadj = Vector{Int}()
    in_Colorsadj = falses(n) # Flag array for O(1) membership testing (Line 51)
    Colorssplit = Vector{Int}()
    
    # Arrays for the SplitUpColour subroutine (Algorithm 3)
    # Preallocate to size n+1 to handle degrees from 0 to n
    numcdeg = zeros(Int, n + 1) 
    f = zeros(Int, n + 1)
    
    
    # Algorithm 2, Line 45
    while !isempty(Srefine)
        r = pop!(Srefine)
        in_stack[r] = false
        
        # 1. Compute colour degrees (Lines 47-54)
        for v in C[r]
            for w in inneighbors(g, v)
                cdeg[w] += 1
                if cdeg[w] == 1
                    push!(A[colour[w]], w)
                end
                
                if !in_Colorsadj[colour[w]]
                    push!(Colorsadj, colour[w])
                    in_Colorsadj[colour[w]] = true
                end
                
                if cdeg[w] > maxcdeg[colour[w]]
                    maxcdeg[colour[w]] = cdeg[w]
                end
            end
        end
        
        # 2. Determine mincdeg and which colors split (Lines 55-65)
        empty!(Colorssplit)
        for c in Colorsadj
            if length(C[c]) != length(A[c])
                mincdeg[c] = 0
            else
                mincdeg[c] = maxcdeg[c]
                for v in A[c]
                    if cdeg[v] < mincdeg[c]
                        mincdeg[c] = cdeg[v]
                    end
                end
            end
            
            if mincdeg[c] < maxcdeg[c]
                push!(Colorssplit, c)
            end
        end
        
        sort!(Colorssplit)
        
        # 3. Split colors (Lines 66-67 / Algorithm 3)
        for s in Colorssplit
            maxcdeg_s = maxcdeg[s]
            
            # Initialize numcdeg for this color class
            for i in 1:maxcdeg_s
                numcdeg[i + 1] = 0
            end
            numcdeg[1] = length(C[s]) - length(A[s]) # numcdeg[0 + 1]
            
            # Count occurrences of each degree
            for v in A[s]
                numcdeg[cdeg[v] + 1] += 1
            end
            
            # Find the majority degree class 'b'
            b = 0
            for i in 1:maxcdeg_s
                if numcdeg[i + 1] > numcdeg[b + 1]
                    b = i
                end
            end
            
            # THE UPGRADE: O(1) membership test instead of searching the stack
            # instack = (s in Srefine) ? 1 : 0
            instack = in_stack[s] ? 1 : 0
            
            # Assign new color labels f[i]
            for i in 0:maxcdeg_s
                if numcdeg[i + 1] >= 1
                    if i == mincdeg[s]
                        f[i + 1] = s
                        if instack == 0 && b != i
                            push!(Srefine, f[i + 1])
                            in_stack[f[i + 1]] = true # Sync state
                        end
                    else
                        k += 1
                        f[i + 1] = k
                        if instack == 1 || i != b
                            push!(Srefine, f[i + 1])
                            in_stack[f[i + 1]] = true # Sync state
                        end
                    end
                end
            end
            
            # move the vertices to their new color classes
            for v in A[s]
                target_color = f[cdeg[v] + 1]
                if target_color != s
                    delete!(C[s], v)           
                    push!(C[target_color], v)  
                    colour[v] = target_color
                end
            end
        end
        
        # 4. Reset attributes for next iteration (Lines 68-73)
        for c in Colorsadj
            for v in A[c]
                cdeg[v] = 0
            end
            maxcdeg[c] = 0
            empty!(A[c])
            in_Colorsadj[c] = false
        end
        empty!(Colorsadj)
    end
    
    return colour
end

canonical_color_refinement (generic function with 1 method)

# Testing the Color-Refinement Algorithm

A simple sanity check: take a graph `G`, relabel its vertices by a random permutation `π`
to get an isomorphic graph `H` (same graph, different vertex numbers). Color refinement is
**canonical**, so the colors must transport through the permutation — i.e.
`colors_H[π(v)] == colors_G[v]` for every vertex. If that holds, the algorithm is working.

In [ ]:
using Random

# Run refinement from the uniform start: every vertex gets color 1, refine class 1.
color_refinement(g::AbstractGraph) = canonical_color_refinement(g, ones(Int, nv(g)), [1])

# Relabel vertices of g by a permutation: vertex v of g becomes vertex perm[v] of the result.
function permute_graph(g, perm)
    h = SimpleGraph(nv(g))
    for e in edges(g)
        add_edge!(h, perm[src(e)], perm[dst(e)])
    end
    return h
end

# Build a graph G, then make H = G with vertices shuffled (isomorphic, different labels).
Random.seed!(1)
G = erdos_renyi(10, 0.4)
perm = randperm(nv(G))          # the relabeling π
H = permute_graph(G, perm)

colorsG = color_refinement(G)
colorsH = color_refinement(H)

# Transport H's colors back onto G's labels: vertex v of G is vertex perm[v] of H.
transported = [colorsH[perm[v]] for v in 1:nv(G)]

println("colors of G            : ", colorsG)
println("colors of H (relabeled): ", colorsH)
println("H's colors mapped to G : ", transported)
println()
println(transported == colorsG ?
        "PASS: same graph, different labels -> identical colors. The algorithm works." :
        "FAIL: colors do not match -> bug.")